## Data Collection
*Refactored to fetch metadata excluding audio features (the audio features API is deprecated).*

Goals for this notebook:
* Pull all raw data by connecting to the Spotify Web API
* Select playlists we want to later analyze.
* Download track, artist, and playlist metadata.

To run this notebook, create the `.env` file based on `.env.sample`.

In [ ]:
from dotenv import load_dotenv
import os
from typing import List, Dict,Any,Iterable
from datetime import date

import spotipy
from spotipy.oauth2 import SpotifyOAuth
from spotipy.oauth2 import SpotifyClientCredentials

import pandas as pd

In [ ]:
load_dotenv()

print(os.getenv("SPOTIPY_CLIENT_ID"))
print(os.getenv("SPOTIPY_CLIENT_SECRET"))
print(os.getenv("SPOTIPY_REDIRECT_URI"))

SCOPE = "playlist-read-private playlist-read-collaborative"
sp_user = spotipy.Spotify(
    auth_manager=SpotifyOAuth(scope=SCOPE)
)

sp_cc = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials()
)

From the API, each playlist is identified by a Spotify playlist ID.

In [300]:
# # manual
# playlist_ids = [
#     "spotify:playlist:xxxxxxxxxxxxxxxxxxxxxx",
#     "spotify:playlist:yyyyyyyyyyyyyyyyyyyyyy",
# ]

def chunked(lst: List[str], size: int) -> Iterable[List[str]]:
    for i in range(0,len(lst),size):
        yield lst[i:i+size]

SNAPSHOT_DATE = date.today().isoformat()

In [301]:
def get_user_playlists(sp_client: spotipy.Spotify) -> pd.DataFrame:
    items = []
    results = sp_client.current_user_playlists(limit=50)
    
    while results:
        for p in results["items"]:
            items.append(
                {
                    "playlist_id": p.get("id"),
                    "name": p.get("name"),
                    "owner": (p.get("owner") or {}).get("id"),
                    "public": p.get("public"),
                    "num_tracks": (p.get("tracks") or {}).get("total"),
                }
            )
        results = sp_client.next(results) if results.get("next") else None

    df = pd.DataFrame(items)
    df = df.dropna(subset=["playlist_id"]).sort_values(["name"]).reset_index(drop=True)
    return df

playlists_df = get_user_playlists(sp_user)

## uncomment to see all playlists
# playlists_df

### Select playlist(s)

In [302]:
SELECT_PLAYLIST_NAMES = [
    "the fray, the script, coldplay, one republic",
    "2000's nostalgia",
    "dramatic piano/violin songs",
    # ... other playlist names ...
]

playlist_ids = playlists_df.loc[playlists_df["name"].isin(SELECT_PLAYLIST_NAMES), "playlist_id"].tolist()

### Collect playlist details
For each playlist we collect track IDs, added date, snapshot date. We filter out missing/local tracks and non-track items

In [303]:
def get_playlist_tracks(sp_client: spotipy.Spotify, playlist_id: str, market: str = "US") -> pd.DataFrame:
    rows = []
    results = sp_client.playlist_items(
        playlist_id,
        market=market,
        additional_types=["track"],
        limit=100,
    )
    while results:
        for item in results.get("items", []):
            track = item.get("track")
            if (
                track is None
                or track.get("id") is None
                or track.get("is_local")
                or track.get("type") != "track"
            ):
                continue

            artists = track.get("artists") or []
            rows.append(
                {
                    "playlist_id": playlist_id,
                    "track_id": track["id"],
                    "track_name": track.get("name"),
                    "artist_id": artists[0].get("id") if artists else None,
                    "artist_name": artists[0].get("name") if artists else None,
                    "added_at": item.get("added_at"),
                    "snapshot_date": SNAPSHOT_DATE,
                }
            )
        results = sp_client.next(results) if results.get("next") else None
    return pd.DataFrame(rows)

playlist_tracks_list = []
for pid in playlist_ids:
    df_pt = get_playlist_tracks(sp_user, pid, market="US")
    # print(pid, "\n# tracks:", len(df_pt))
    playlist_tracks_list.append(df_pt)

if not playlist_tracks_list or all(df.empty for df in playlist_tracks_list):
    raise ValueError("no tracks collected. please confirm playlist IDs are valid/contain playable tracks.")

playlist_tracks_df = pd.concat(playlist_tracks_list, ignore_index=True)
# playlist_tracks_df.head()

### Track metadata

We will fetch metadata from each unique ID including:
- popularity
- duration
- album
- release date

The `tracks()` endpoint can take up to 50 IDs per request.

In [304]:
unique_track_ids = playlist_tracks_df["track_id"].dropna().unique().tolist()
print("Unique track IDs:", len(unique_track_ids))

Unique track IDs: 141


In [305]:
def get_track_metadata(sp_client: spotipy.Spotify, track_ids: List[str]) -> pd.DataFrame:
    rows = []
    for chunk in chunked(track_ids, size=50):
        res = sp_client.tracks(chunk)
        for t in res.get("tracks", []):
            if t is None or t.get("id") is None:
                continue
            album = t.get("album") or {}
            artists = t.get("artists") or []
            rows.append(
                {
                    "track_id": t["id"],
                    "track_name": t.get("name"),
                    "track_popularity": t.get("popularity"),
                    "duration_ms":t.get("duration_ms"),
                    "explicit": t.get("explicit"),
                    "album_id": album.get("id"),
                    "album_name": album.get("name"),
                    "release_date": album.get("release_date"),
                    "release_date_precision": album.get("release_date_precision"),
                    "artist_id": artists[0].get("id") if artists else None,
                    "artist_name": artists[0].get("name") if artists else None,
                }
            )
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["track_id"]).reset_index(drop=True)
    return df

track_meta_df = get_track_metadata(sp_cc, unique_track_ids)
track_meta_df.drop(columns=["track_id", "album_id", "artist_id"], errors="ignore").head()

,track_name,track_popularity,duration_ms,explicit,album_name,release_date,release_date_precision,artist_name
0,Shattered (Turn The Car Around),65,255493,False,All Sides (Deluxe Edition),2008-07-15,day,O.A.R.
1,Stop And Stare,67,223853,False,Dreaming Out Loud,2007-01-01,day,OneRepublic
2,Don't Know Why,80,186251,False,Come Away With Me (Super Deluxe Edition),2002-02-26,day,Norah Jones
3,Breakeven,84,261426,False,The Script,2008-07-14,day,The Script
4,I'm Yours,84,242946,False,We Sing. We Dance. We Steal Things. (Bonus Tra...,2008-05-01,day,Jason Mraz


### Artist metadata

Now we fetch artist catalog metadata for each unique artist ID, which includes:
- artist_popularity
- artist_followers
- genres

In [306]:
unique_artist_ids = track_meta_df["artist_id"].dropna().unique().tolist()

# the `artists()` endpoint also supports up to 50 IDs.
print("# artist IDs:", len(unique_artist_ids))

# artist IDs: 101


In [307]:
def get_artist_metadata(sp_client: spotipy.Spotify, artist_ids: List[str]) -> pd.DataFrame:
    rows = []
    artist_ids = [a for a in set(artist_ids) if isinstance(a, str) and a]

    for chunk in chunked(artist_ids, size=50):
        res = sp_client.artists(chunk)
        for a in res.get("artists", []):
            if a is None or a.get("id") is None:
                continue

            rows.append(
                {
                    "artist_id": a["id"],
                    "artist_name": a.get("name"),
                    "artist_popularity": a.get("popularity"),
                    "artist_followers": (a.get("followers") or {}).get("total"),
                    "genres": " | ".join(a.get("genres") or []),
                }
            )
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["artist_id"]).reset_index(drop=True)
    return df

artist_meta_df = get_artist_metadata(sp_cc, unique_artist_ids)
# artist_meta_df.head()

### Combine track + artist features into a single track feature table
This is what we will use as the base feature matrix

In [308]:
tracks_features_df = track_meta_df.merge(
    artist_meta_df,
    on="artist_id",
    how="left",
    suffixes=("","_artist")
)
# tracks_features_df.head()

In [309]:
def get_playlist_followers(sp_client: spotipy.Spotify, playlist_id: str) -> Dict[str, Any]:
    pl = sp_client.playlist(playlist_id, fields="id,name,followers.total")
    return {
        "playlist_id": pl.get("id"),
        "playlist_name": pl.get("name"),
        "followers": (pl.get("followers") or {}).get("total"),
        "snapshot_date": SNAPSHOT_DATE,
    }
playlist_followers_df = pd.DataFrame([get_playlist_followers(sp_user, pid) for pid in playlist_ids])
# playlist_followers_df.drop(columns=["playlist_id"], errors="ignore").head()

### Save outputs

In [310]:
out_dir = "data/raw"
os.makedirs(out_dir, exist_ok=True)

playlist_tracks_path = os.path.join(out_dir, f"playlist_tracks_{SNAPSHOT_DATE}.csv")
playlist_followers_path = os.path.join(out_dir, f"playlist_followers_{SNAPSHOT_DATE}.csv")
tracks_features_path = os.path.join(out_dir, "tracks_features.csv")
artists_features_path = os.path.join(out_dir, "artists_features.csv")

playlist_tracks_df.to_csv(playlist_tracks_path,index=False)
playlist_followers_df.to_csv(playlist_followers_path,index=False)
tracks_features_df.to_csv(tracks_features_path,index=False)
artist_meta_df.to_csv(artists_features_path,index=False)

print("dataframe shapes:")
print("playlist_tracks_df:",playlist_tracks_df.shape)
print("playlist_followers_df:", playlist_followers_df.shape)
print("tracks_features_df:", tracks_features_df.shape)
print("artist_meta_df:", artist_meta_df.shape)

## check for null values
# print("tracks_features_df missing artist_popularity:", tracks_features_df["artist_popularity"].isna().mean())
# print("tracks_features_df missing artist_followers:", tracks_features_df["artist_followers"].isna().mean())
# print("tracks_features_df missing genres:", tracks_features_df["genres"].isna().mean())

dataframe shapes:
playlist_tracks_df: (147, 7)
playlist_followers_df: (1, 4)
tracks_features_df: (141, 15)
artist_meta_df: (101, 5)
